In [1]:
from sl_analytics.db import query

In [2]:
devices = query("SELECT id, device_key, display_name, measurement_source, timezone FROM devices ORDER BY id LIMIT 20")
devices

,id,device_key,display_name,measurement_source,timezone
0,1,1101,CNC 1101,power,America/Santiago
1,2,1102,CNC 1102,power,America/Santiago
2,3,1103,CNC 1103,power,America/Santiago
3,4,1104,CNC 1104,power,America/Santiago
4,5,1105,CNC 1105,power,America/Santiago
5,6,1106,CNC 1106,power,America/Santiago
6,7,1107,CNC 1107,power,America/Santiago
7,9,Rep10,Rep10,power,America/Santiago
8,10,Rep44,Rep44,power,America/Santiago
9,11,Rep77,Rep77,power,America/Santiago


In [3]:
device_id = int(devices.iloc[0]["id"])
device_id

1

In [4]:
query(
    "SELECT day, load_minutes, off_minutes "
    "FROM device_daily_facts "
    "WHERE device_id = :d "
    "ORDER BY day DESC LIMIT 30",
    {"d": device_id},
)

,day,load_minutes,off_minutes
0,2026-04-21,445.97565,24.433332
1,2026-04-20,794.50000,0.883333
2,2026-04-17,317.98334,11.983334
3,2026-04-16,575.71670,0.133333
4,2026-04-15,569.96670,0.416667
5,2026-04-14,564.68335,6.016667
6,2026-04-13,589.18335,5.216667
7,2026-04-10,319.63333,10.333333
8,2026-04-09,572.55000,0.133333
9,2026-04-08,587.55000,0.150000


## Configuración de umbrales por dispositivo

Umbrales de histéresis (`on_threshold_w` / `off_threshold_w`) que usa el proceso para decidir LOAD o OFF. Unimos `device_power_state_config` con `devices` para mostrar el nombre descriptivo en lugar de `device_id`.

In [12]:
df = query(    "SELECT d.display_name, "
    "       c.on_threshold_w, "
    "       c.power_column "
    "FROM device_power_state_config c "
    "JOIN devices d ON d.id = c.device_id "
    "ORDER BY d.display_name")

df = df.rename(columns={

    "display_name": "Dispositivo",

    "on_threshold_w": "Umbral Encendido (W)",

    "power_column": "Columna Potencia"

})

In [13]:
df

,Dispositivo,Umbral Encendido (W),Columna Potencia
0,CNC 1101,500.0,total_active_power
1,CNC 1102,520.0,total_active_power
2,CNC 1103,480.0,total_active_power
3,CNC 1104,600.0,total_active_power
4,CNC 1105,700.0,total_active_power
5,CNC 1106,220.0,total_active_power
6,CNC 1107,650.0,total_active_power
7,CNC S1,5.0,total_current
8,FR S1,60.0,total_current
9,INF TBX,21.7,total_current


## Mediciones de un dispositivo dentro del horario (lunes 13 a viernes 17)

Trae las lecturas de `power_measurements` (`total_active_power` y `total_current`) de **un solo dispositivo** en el rango lunes 2026-04-13 a viernes 2026-04-17, y **solo las que caen dentro del horario de trabajo** (`i.on_schedule = true` en `device_state_intervals`).

Cada fila viene con:

- `classification`: `LOAD` u `OFF` según el intervalo derivado.
- `on_schedule`: siempre `true` en este resultado (lo usamos como filtro).
- `on_threshold_w` / `off_threshold_w` / `power_column`: umbrales de histéresis de ese dispositivo.

Cambiá `company_name` y `device_id` en la celda de parámetros para analizar otra combinación. La fecha fin (`to_ts`) es **exclusiva**, así que `'2026-04-18 00:00:00-03'` incluye todo el viernes 17.

## Mediciones de una empresa en la semana 13–17 de abril

Obtiene las lecturas de `power_measurements` (`total_active_power` y `total_current`) para todos los dispositivos de una empresa, en el rango del lunes 2026-04-13 al viernes 2026-04-17. Cada fila incluye:

- `classification`: `LOAD` o `OFF` según el intervalo derivado (`device_state_intervals`). Si una lectura aún no se ha procesado, queda `NULL`.
- `on_threshold_w` / `off_threshold_w` / `power_column`: los umbrales de histéresis configurados para ese dispositivo.

Cambia `company_name` al nombre de la empresa que quieras analizar. La fecha de fin (`to_ts`) es **exclusiva**, por lo que `'2026-04-18 00:00:00-03'` incluye todo el viernes 17.

In [16]:
# Por default tomamos la primera empresa. Cambialo al nombre que necesites.
company_name = query("SELECT name FROM companies ORDER BY id LIMIT 1").iloc[0]["name"]

# Por default tomamos el primer dispositivo de esa empresa. Cambialo al que quieras analizar.
device_row = query(
    "SELECT d.id, d.display_name "
    "FROM devices d JOIN companies c ON c.id = d.company_id "
    "WHERE c.name = :company ORDER BY d.id LIMIT 1",
    {"company": company_name},
).iloc[0]
device_id = int(device_row["id"])
device_display = device_row["display_name"]
print(f"Empresa: {company_name}  |  Dispositivo: {device_display} (id={device_id})")

from_ts = "2026-04-13 00:00:00-03"   # lunes 13 inclusive
to_ts   = "2026-04-18 00:00:00-03"   # sábado 18 exclusive (cubre todo el viernes 17)

Empresa: Repairco  |  Dispositivo: CNC 1101 (id=1)


In [17]:
sql = """
SELECT
    pm.time,
    TO_CHAR(pm.time, 'DD-MM-YYYY') AS fecha,
    TO_CHAR(pm.time, 'HH24:MI:SS') AS hora,
    d.display_name,
    pm.total_active_power,
    pm.total_current,
    i.state AS classification,
    i.on_schedule,
    psc.on_threshold_w,
    psc.off_threshold_w,
    psc.power_column
FROM power_measurements pm
JOIN devices d ON d.id = pm.device_id
LEFT JOIN device_power_state_config psc ON psc.device_id = d.id
JOIN device_state_intervals i
       ON i.device_id  = pm.device_id
      AND i.source     = 'power'
      AND i.start_time <= pm.time
      AND (i.end_time IS NULL OR i.end_time > pm.time)
WHERE pm.device_id  = :device
  AND i.on_schedule = TRUE
  AND pm.time >= :f
  AND pm.time <  :t
ORDER BY pm.time
LIMIT 50000;
"""

mediciones = query(sql, {"device": device_id, "f": from_ts, "t": to_ts})
print(f"Filas: {len(mediciones):,}")
mediciones.head(20)

Filas: 50,000


,time,fecha,hora,display_name,total_active_power,total_current,classification,on_schedule,on_threshold_w,off_threshold_w,power_column
0,2026-04-13 11:30:00.499000+00:00,13-04-2026,07:30:00,CNC 1101,320.605,3.340,OFF,True,500.0,430.0,total_active_power
1,2026-04-13 11:30:02.277000+00:00,13-04-2026,07:30:02,CNC 1101,321.261,3.335,OFF,True,500.0,430.0,total_active_power
2,2026-04-13 11:30:04.626000+00:00,13-04-2026,07:30:04,CNC 1101,320.033,3.333,OFF,True,500.0,430.0,total_active_power
3,2026-04-13 11:30:06.975000+00:00,13-04-2026,07:30:06,CNC 1101,319.999,3.337,OFF,True,500.0,430.0,total_active_power
4,2026-04-13 11:30:08.277000+00:00,13-04-2026,07:30:08,CNC 1101,320.874,3.344,OFF,True,500.0,430.0,total_active_power
5,2026-04-13 11:30:10.276000+00:00,13-04-2026,07:30:10,CNC 1101,321.631,3.351,OFF,True,500.0,430.0,total_active_power
6,2026-04-13 11:30:12.278000+00:00,13-04-2026,07:30:12,CNC 1101,322.237,3.354,OFF,True,500.0,430.0,total_active_power
7,2026-04-13 11:30:14.539000+00:00,13-04-2026,07:30:14,CNC 1101,320.639,3.345,OFF,True,500.0,430.0,total_active_power
8,2026-04-13 11:30:16.857000+00:00,13-04-2026,07:30:16,CNC 1101,321.160,3.340,OFF,True,500.0,430.0,total_active_power
9,2026-04-13 11:30:18.270000+00:00,13-04-2026,07:30:18,CNC 1101,320.773,3.339,OFF,True,500.0,430.0,total_active_power


# Guardar DF en equipo local
1. Como CSV

In [8]:
mediciones.to_csv("mediciones.csv", index=False)